# 01. Análise exploratória

**Objetivo:** entender a base antes de modelar, responder quais variáveis separam bons e maus pagadores e justificar as decisões de tratamento.

**Importante:** a análise é feita **apenas na base de treino**. Olhar o teste antes da modelagem é uma forma sutil de vazamento de informação, porque decisões tomadas olhando o teste deixam a avaliação final otimista.

Antes de rodar, gere os arquivos de treino e teste no terminal:
```bash
python -m src.data
```

## Dicionário de dados

Cada linha é um cliente de cartão de crédito em Taiwan. As variáveis descrevem os dados cadastrais e o comportamento de **abril a setembro de 2005**; o alvo indica se o cliente ficou inadimplente em **outubro**. Valores em NT$ (novo dólar taiwanês).

O sufixo numérico indica o mês: **1 = setembro** (mais recente) até **6 = abril**.

| Coluna | Tipo | Descrição |
|---|---|---|
| `credit_limit` | quantitativa | Limite de crédito do cartão |
| `sex` | nominal | 1 masculino, 2 feminino |
| `education` | categórica | 1 pós-graduação, 2 universidade, 3 ensino médio, 4 outros |
| `marriage` | nominal | 1 casado, 2 solteiro, 3 outros |
| `age` | quantitativa | Idade em anos |
| `pay_status_1` a `pay_status_6` | código | Situação do pagamento: -2 sem uso, -1 pagou tudo, 0 rotativo, 1 a 8 meses de atraso |
| `bill_amt_1` a `bill_amt_6` | quantitativa | Valor da fatura; negativo = crédito a favor do cliente |
| `pay_amt_1` a `pay_amt_6` | quantitativa | Valor efetivamente pago no mês |
| `default` | alvo | 1 = inadimplente em outubro, 0 = pagou |

Os nomes originais da UCI (`LIMIT_BAL`, `PAY_0`, `BILL_AMT1`...) foram trocados no download por nomes mais claros. O mapeamento está em `scripts/download_data.py`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Permite importar o pacote src/ a partir da pasta notebooks/
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.data import TRAIN_CSV, TARGET, CATEGORICAL, NUMERIC, BILL_AMT, FEATURES, load_raw

# Cores: azul para adimplentes, laranja para inadimplentes
COR_BOM, COR_MAU = "#2a78d6", "#eb6834"
plt.rcParams.update({
    "figure.figsize": (8, 4),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
})
pd.set_option("display.float_format", "{:,.2f}".format)

train = pd.read_csv(TRAIN_CSV)
taxa_geral = train[TARGET].mean()
print(f"Treino: {train.shape[0]:,} linhas, {train.shape[1]} colunas")
print(f"Taxa de inadimplência: {taxa_geral:.2%}")

## 1. Visão geral das variáveis

Estatísticas descritivas **apenas das variáveis quantitativas**. As categóricas (`sex`, `education`, `marriage`) são analisadas por frequência e taxa de inadimplência nas seções 2 e 3, porque média de um código não tem significado.

Atenção a três pontos: a escala muito diferente entre variáveis (idade em anos, valores em dólares taiwaneses), a assimetria dos valores monetários e a existência de **faturas negativas** (`bill_amt_*`), que indicam crédito a favor do cliente.

In [ ]:
# Só as variáveis quantitativas: média e desvio não fazem sentido para categorias
train[NUMERIC].describe().T[["mean", "std", "min", "50%", "max"]]

## 2. Categorias fora da documentação

A documentação da UCI define `education` de 1 a 4 e `marriage` de 1 a 3. A base original traz códigos que não existem nessa definição. Na limpeza (`src/data.py`), eles foram agrupados em "outros". A célula abaixo mostra a base **antes** da limpeza, para justificar a decisão.

In [ ]:
raw = load_raw()
print("education (original):")
print(raw["education"].value_counts().sort_index().to_string())
print()
print("marriage (original):")
print(raw["marriage"].value_counts().sort_index().to_string())

## 3. Taxa de inadimplência por categoria

Para cada categoria: quantos clientes há e qual a taxa de inadimplência. Compare com a taxa geral.

Legenda: `sex` 1 masculino, 2 feminino. `education` 1 pós-graduação, 2 universidade, 3 ensino médio, 4 outros. `marriage` 1 casado, 2 solteiro, 3 outros.

In [ ]:
def taxa_por_grupo(df, coluna):
    tabela = df.groupby(coluna)[TARGET].agg(clientes="size", taxa_inadimplencia="mean")
    tabela["diferenca_vs_geral"] = tabela["taxa_inadimplencia"] - taxa_geral
    return tabela

for col in CATEGORICAL:
    print(f"--- {col} ---")
    print(taxa_por_grupo(train, col).to_string())
    print()

## 4. Histórico de pagamento recente (`pay_status_1`)

`pay_status_1` é a situação do pagamento no mês mais recente:
- `-2`: sem consumo no mês
- `-1`: fatura paga integralmente
- `0`: pagou o mínimo (crédito rotativo)
- `1` a `8`: meses de atraso

Em risco de crédito, o comportamento recente costuma ser o sinal mais forte. O gráfico mostra se isso vale aqui.

In [ ]:
tabela_pay1 = taxa_por_grupo(train, "pay_status_1")
print(tabela_pay1.to_string())

fig, ax = plt.subplots()
ax.bar(tabela_pay1.index.astype(str), tabela_pay1["taxa_inadimplencia"], color=COR_MAU, width=0.6)
ax.axhline(taxa_geral, color="gray", linestyle="--", linewidth=1)
ax.text(-0.5, taxa_geral + 0.01, f"taxa geral {taxa_geral:.0%}", ha="left", color="gray")
ax.set_xlabel("pay_status_1 (situação do último mês)")
ax.set_ylabel("Taxa de inadimplência")
ax.set_title("Inadimplência por situação de pagamento no último mês")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
plt.show()

## 5. Limite de crédito por classe

Se as distribuições forem diferentes entre adimplentes e inadimplentes, o limite ajuda a separar as classes. Cada curva mostra a porcentagem dos clientes **da própria classe** em cada faixa, porque as classes têm tamanhos diferentes.

In [ ]:
fig, ax = plt.subplots()
bins = np.linspace(0, train["credit_limit"].quantile(0.99), 40)
for classe, cor, rotulo in [(0, COR_BOM, "Adimplente"), (1, COR_MAU, "Inadimplente")]:
    valores = train.loc[train[TARGET] == classe, "credit_limit"]
    pesos = np.ones(len(valores)) / len(valores)  # cada classe soma 100%
    ax.hist(valores, bins=bins, weights=pesos,
            histtype="step", linewidth=2, color=cor, label=rotulo)
ax.set_xlabel("Limite de crédito (NT$, até o percentil 99)")
ax.set_ylabel("% dos clientes da classe")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.1%}"))
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1000:.0f} mil"))
ax.set_title("Distribuição do limite de crédito por classe")
ax.legend()
plt.show()

print(train.groupby(TARGET)["credit_limit"].median().rename("mediana do limite"))

## 6. Correlação entre as faturas (multicolinearidade)

As faturas de meses seguidos tendem a ser muito parecidas. Correlação alta entre variáveis explicativas não atrapalha a previsão, mas deixa os **coeficientes da regressão logística instáveis e difíceis de interpretar**. Isso vai pesar na escolha da regularização no dia 3.

In [ ]:
train[BILL_AMT].corr().round(2)

## 7. Poder de separação de cada variável: Information Value (IV)

O IV é a métrica clássica de seleção de variáveis em modelos de crédito. Para cada faixa da variável, compara a proporção de bons e maus pagadores (o *Weight of Evidence*, WoE) e soma essas diferenças.

Leitura usual do mercado:

| IV | Poder preditivo |
|---|---|
| < 0,02 | inútil |
| 0,02 a 0,10 | fraco |
| 0,10 a 0,30 | médio |
| 0,30 a 0,50 | forte |
| > 0,50 | suspeito (checar vazamento) |

In [ ]:
def information_value(df, coluna, n_faixas=10):
    x = df[coluna]
    # Variáveis com poucos valores ficam como estão; as contínuas viram decis
    faixas = x if x.nunique() <= 12 else pd.qcut(x, q=n_faixas, duplicates="drop")
    contagem = pd.crosstab(faixas, df[TARGET])
    eps = 0.5  # suavização para evitar divisão por zero em faixas sem maus
    dist_bons = (contagem[0] + eps) / (contagem[0].sum() + eps)
    dist_maus = (contagem[1] + eps) / (contagem[1].sum() + eps)
    woe = np.log(dist_bons / dist_maus)
    return ((dist_bons - dist_maus) * woe).sum()

def classificar_iv(iv):
    if iv < 0.02: return "inútil"
    if iv < 0.10: return "fraco"
    if iv < 0.30: return "médio"
    if iv < 0.50: return "forte"
    return "suspeito"

ranking_iv = (
    pd.DataFrame({"iv": {col: information_value(train, col) for col in FEATURES}})
    .sort_values("iv", ascending=False)
)
ranking_iv["poder"] = ranking_iv["iv"].map(classificar_iv)
ranking_iv.style.format({"iv": "{:.3f}"})

## 8. Conclusões

Preencha com o que você observou. Estas conclusões vão para o README e são o que você vai contar na entrevista.

1. **Desbalanceamento:** a taxa de inadimplência é de ___%. Por isso, a avaliação vai usar ___ em vez de acurácia.
2. **Variável mais forte:** ___, com IV de ___. Faz sentido de negócio porque ___.
3. **Categorias corrigidas:** ___.
4. **Multicolinearidade:** as faturas têm correlação de até ___ entre si. Consequência para a regressão logística: ___.
5. **Variáveis fracas:** ___. Decisão: manter no modelo por enquanto, porque ___.